Databricks notebook source
# TITLE: Delta Lake Upserts (MERGE) - Enterprise Pattern
##  DESCRIPTION: Demonstrates how to handle changing data without creating duplicates.

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from delta.tables import DeltaTable

In [0]:
# ---------------------------------------------------------
# STEP 1: CREATE THE INITIAL DATA (The "Target" Data Lake)
# ---------------------------------------------------------
print("Step 1: Creating initial Customer Data Lake...")

schema = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("Address", StringType(), True),
    StructField("Status", StringType(), True)
])

# Day 1 Data: John and Sarah exist in our database
initial_data = [
    (101, "John Doe", "123 Old Street, NY", "Active"),
    (102, "Sarah Smith", "456 Oak Avenue, CA", "Active")
]

target_df = spark.createDataFrame(data=initial_data, schema=schema)

# Write to Delta Lake
target_table_path = "/tmp/enterprise_patterns/customer_delta_lake"
target_df.write.format("delta").mode("overwrite").save(target_table_path)

display(spark.read.format("delta").load(target_table_path))



In [0]:
# ---------------------------------------------------------
# STEP 2: NEW DAILY UPDATES ARRIVE (The "Source" Data)
# ---------------------------------------------------------
print("Step 2: New daily batch arrives (Updates & New Customers)...")

# Day 2 Data: John moved to a New Street. Mike is a brand new customer.
new_batch_data = [
    (101, "John Doe", "999 NEW STREET, NY", "Active"), # UPDATE
    (103, "Mike Johnson", "789 Pine Road, TX", "Active") # INSERT
]

source_df = spark.createDataFrame(data=new_batch_data, schema=schema)
display(source_df)


In [0]:
# ---------------------------------------------------------
# STEP 3: THE ENTERPRISE MERGE (UPSERT)
# ---------------------------------------------------------
print("Step 3: Executing Delta MERGE to prevent duplicates...")

# Load the target Delta Table
delta_target = DeltaTable.forPath(spark, target_table_path)

# Execute the Upsert Logic
(delta_target.alias("target")
    .merge(
        source_df.alias("source"),
        "target.CustomerID = source.CustomerID" # The matching key
    )
    .whenMatchedUpdateAll() # If ID exists, update all columns
    .whenNotMatchedInsertAll() # If ID doesn't exist, insert the new row
    .execute()
)


In [0]:
# ---------------------------------------------------------
# STEP 4: VERIFY THE CLEAN DATA LAKE
# ---------------------------------------------------------
print("Step 4: Final Data Lake State (Notice John is updated, Mike is added, NO duplicates!)")

final_df = spark.read.format("delta").load(target_table_path).orderBy("CustomerID")
display(final_df)

# Clean up (Optional)
# dbutils.fs.rm(target_table_path, True)